In [2]:
!pip install -q langchain langchain-google-genai langchain-core python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.7 MB/s eta 0:00:00


In [4]:
!pip install -U langchain-groq groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00


In [5]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b (Small/Fast) to demonstrate logic failures
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

Enter your Groq API Key: ··········


In [6]:
question = "Priya has 10 pencils . she buys 2 more packets of apsara pencils. Each packet has 6 pencils. How many does she have now?"
prompt_standard = f"Answer this question: {question}"
print("--- STANDARD (Llama3.1-8b) ---")
print(llm.invoke(prompt_standard).content)

--- STANDARD (Llama3.1-8b) ---
To find out how many pencils Priya has now, we need to add the pencils she already had to the pencils she bought.

Priya initially had 10 pencils. 
She bought 2 packets of pencils, each containing 6 pencils. 
So, she bought 2 x 6 = 12 pencils.

Now, let's add the pencils she already had to the pencils she bought: 
10 (initial pencils) + 12 (new pencils) = 22 pencils.

Priya now has 22 pencils.


In [7]:
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- Chain of Thought (Llama3.1-8b) ---")
print(llm.invoke(prompt_cot).content)


--- Chain of Thought (Llama3.1-8b) ---
To find out how many pencils Priya has now, we need to follow these steps:

1. Priya already has 10 pencils.
2. She buys 2 more packets of Apsara pencils. Each packet has 6 pencils.
3. So, she buys 2 x 6 = 12 more pencils.
4. Now, we add the pencils she already had (10) to the pencils she bought (12). 
5. 10 + 12 = 22

So, Priya now has 22 pencils.


**Part 3b: Tree of Thoughts (ToT) & Graph of Thoughts (GoT)**

In [8]:
!pip install python-dotenv --upgrade --quiet langchain langchain-groq

from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 11.6 MB/s eta 0:00:00


In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "What should I start eating when I am going to Gym?"

# Step 1: The Branch Generator
prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# Step 2: The Judge
prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'

    1: {sol1}
    2: {sol2}
    3: {sol3}

    Act as a Nutrionist. Pick the most sustainable one (not bribery) and explain why.
    """
)

# Chain: Input -> Branches -> Judge -> Output
tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
As a nutritionist, I would recommend **Solution 1: Pre-Workout Energy Bites** as the most sustainable option. Here's why:

1. **Sustained Energy**: These energy bites are made with a combination of complex carbohydrates (oats), healthy fats (nut butter), and protein (protein powder). This blend provides sustained energy throughout your workout, reducing the need for mid-workout snacking or energy crashes.

2. **Convenience**: Energy bites are easy to prepare, pack, and consume on-the-go. They're perfect for a pre-workout snack, and you can take them with you to the gym.

3. **Customization**: The recipe allows for customization with various nut butters, dried fruits, and protein powders. This means you can experiment with different flavors and ingredients to find the perfect combination for your taste preferences and dietary needs.

4. **Portion Control**: Each energy bite is approximately 1-inch in diameter, making it easy to control the portion s

In [11]:
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)

# 2. The Aggregator (Convergence)
prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}

    Your task: Create a new Mega-Movie that combines the TECHNOLOGY of Sci-Fi, the PASSION of Romance, and the FEAR of Horror.
    Write one paragraph.
    """
)

# 3. The Chain
got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
print(got_chain.invoke("Willy Wonka & the chocolate factory"))

--- Graph of Thoughts (GoT) Result ---
In 'Wonka's Sweet Reckoning,' a dystopian future where a mysterious substance known as 'Echo' has been discovered, holding the key to unlocking humanity's true potential, Willy Wonka's advanced chocolate factory serves as the testing ground for this enigmatic substance. As Violet Beauregarde, a brilliant and determined chocolatier, falls deeply in love with the charismatic Wonka, she becomes increasingly entangled in the dark secrets of his past. However, when Wonka's obsession with Echo reaches a fever pitch, his creations begin to take on a life of their own, manifesting as twisted, sentient beings that prey on the Oompa Loompas, forcing Charlie Bucket to navigate a perilous world of sweet-toothed terror, where the lines between love, technology, and terror are blurred, and the very fabric of reality hangs in the balance.
